# SMA Crossover Backtest (db_accessor_client + indicator_engine)

This notebook fetches candles via `db_accessor_client`, computes SMAs with `indicator_engine`, and runs a simple long/flat crossover strategy.

In [10]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd

# Make local libs and backtester src importable
repo_root = Path.cwd().resolve().parent
sys.path.insert(0, str(Path.cwd().resolve()))
sys.path.insert(0, str(repo_root / "libs" / "db_accessor_client"))
sys.path.insert(0, str(repo_root / "libs" / "indicator_engine"))

from db_accessor_client import DatabaseAccessorClient, DatabaseAccessorClientError
from indicator_engine import run
from src.portfolio import Portfolio
from src.signals import crossover, crossunder

os.environ.setdefault("DATABASE_ACCESSOR_HOST", "localhost")
os.environ.setdefault("DATABASE_ACCESSOR_PORT", "8000")

pd.set_option("display.max_rows", 10)
pd.set_option("display.max_columns", 20)

In [11]:
# Strategy / data parameters
SYMBOL = "EURUSD"
TIMEFRAME = "M15"
LIMIT = 100000
FAST_WINDOW = 20
SLOW_WINDOW = 30

try:
    with DatabaseAccessorClient() as client:
        candles = client.get_candles(symbol=SYMBOL, timeframe=TIMEFRAME, limit=LIMIT)
except DatabaseAccessorClientError as exc:
    host = os.getenv("DATABASE_ACCESSOR_HOST", "localhost")
    port = os.getenv("DATABASE_ACCESSOR_PORT", "8000")
    raise RuntimeError(f"Could not reach database-accessor-api at {host}:{port}.") from exc

if candles.empty:
    raise ValueError("No candles returned from database-accessor-api.")

# Keep canonical OHLCV order expected by most indicator workflows
candles = candles[["open", "high", "low", "close", "volume"]].copy()
candles.tail()

,open,high,low,close,volume
timestamp,,,,,
2026-04-06 14:30:00+00:00,1.15655,1.15715,1.15534,1.15622,2284.0
2026-04-06 14:45:00+00:00,1.15622,1.15628,1.15426,1.15431,2672.0
2026-04-06 15:00:00+00:00,1.15429,1.15436,1.15373,1.15378,2169.0
2026-04-06 15:15:00+00:00,1.15375,1.15422,1.15354,1.15401,1619.0
2026-04-06 15:30:00+00:00,1.15401,1.15417,1.15365,1.15413,778.0


In [12]:
sma_fast = run("sma", candles, params={"window": FAST_WINDOW, "source": "close"})
sma_slow = run("sma", candles, params={"window": SLOW_WINDOW, "source": "close"})

In [13]:
df = candles[["open", "close"]].copy()
df["sma_fast"] = sma_fast["sma"]
df["sma_slow"] = sma_slow["sma"]

buy_signals = crossover(df[["sma_fast"]], df[["sma_slow"]])
sell_signals = crossunder(df[["sma_fast"]], df[["sma_slow"]])

In [14]:
portfolio = Portfolio.from_signals(df[["open"]], buy_signals, sell_signals)
stats = portfolio.get_stats()

In [15]:
preview = df[["close", "sma_fast", "sma_slow"]].copy()
preview["buy_signal"] = buy_signals.iloc[:, 0]
preview["sell_signal"] = sell_signals.iloc[:, 0]
preview["position"] = portfolio.positions.iloc[:, 0]
preview.tail(20)

,close,sma_fast,sma_slow,buy_signal,sell_signal,position
timestamp,,,,,,
2026-04-06 10:45:00+00:00,1.15479,1.154681,1.153886,False,False,True
2026-04-06 11:00:00+00:00,1.15430,1.154762,1.153962,False,False,True
2026-04-06 11:15:00+00:00,1.15431,1.154860,1.154042,False,False,True
2026-04-06 11:30:00+00:00,1.15437,1.154946,1.154120,False,False,True
2026-04-06 11:45:00+00:00,1.15457,1.155008,1.154207,False,False,True
...,...,...,...,...,...,...
2026-04-06 14:30:00+00:00,1.15622,1.155179,1.155278,False,False,False
2026-04-06 14:45:00+00:00,1.15431,1.155142,1.155307,False,False,False
2026-04-06 15:00:00+00:00,1.15378,1.155075,1.155259,False,False,False


In [16]:
stats["Profit (%)"].sum()

np.float64(-2.7422617762214774)